# Mission 3 환자 증상 인식 — KcELECTRA 학습

**모델**: `beomi/KcELECTRA-base-v2022` + HF 기본 분류 head (`AutoModelForSequenceClassification`, 9개 증상 멀티라벨)

| 항목 | 설정 |
|---|---|
| 입력 | 대화 전사 텍스트(`text`)만 시간순으로 이어붙임 (화자 태그 미사용), max_length 512 |
| 출력 | 9개 증상 멀티라벨 |
| Loss | BCEWithLogitsLoss (pos_weight 없음, `CONFIG_NAME = "base"`) |
| 최적화 | AdamW lr 3e-5, weight decay 0.01, warmup 6% + linear decay, fp16, batch 16 |
| 학습 | epoch 상한 5, early stopping patience 2, best epoch = Validation Macro F1 |
| 판정 | **threshold 0.5 고정** (모든 클래스 동일) |
| seed | 42, 123, 2026 |

- 학습은 Training(서울) 29,200건만 사용, Validation은 best epoch 선정에만 사용 (학습 미사용)
- 결과 (Validation Macro F1): seed 42 **0.6083** / 123 **0.6082** / 2026 **0.6056**, 3-seed 확률 평균 **0.6136**
- 최종 제출: 이 모델 3 seed + KLUE-RoBERTa 3 seed 확률 평균 앙상블 (Macro F1 0.6543)
- 학습 라벨 순서는 `CONFIG["symptom_labels"]` 순서이며, 제출용 체크포인트(`ckpt_m3/kcelectra`)는 출력층을 팀 공통 순서(고열~호흡곤란)로 재배열해 저장함
- 이어하기: 매 epoch 끝에 학습 상태를 Drive에 저장하고, 끊긴 뒤 다시 실행하면 마지막 epoch부터 이어서 학습

### 0. GPU 확인

In [1]:
!nvidia-smi


Tue Oct 13 11:26:43 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   48C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

### 1. 환경 설정

In [2]:
# ===== 환경 설정 =====
IS_DRIVE = True  # Google Drive 마운트를 쓰면 True

if IS_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')

!pip install -q transformers scikit-learn

import os, json, shutil, random, time, uuid
os.environ["TOKENIZERS_PARALLELISM"] = "false"  # DataLoader worker에서 토크나이저 경고/교착 방지
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import f1_score, accuracy_score
import transformers
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
GPU_NAME = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"
print(f"device={device}, GPU={GPU_NAME}, torch={torch.__version__}, transformers={transformers.__version__}")

# 실험 전체에서 공통으로 쓰는 seed 목록 (팀 공통 규칙)
SEEDS = [42, 123, 2026]

DRIVE_ROOT = ("/content/drive/MyDrive/projects/AI-DCC/mission1-data" if IS_DRIVE
              else os.path.join(os.path.expanduser("~"), "Downloads", "mission1-data"))

# 백본 후보 (모두 공개 pretrained, FAQ Q1 허용). 백본마다 배치 크기/학습률을 따로 둠
BACKBONES = {
    "kcelectra":          {"hf_id": "beomi/KcELECTRA-base-v2022",                 "batch_size": 16, "lr": 3e-5},
    "klue_roberta_base":  {"hf_id": "klue/roberta-base",                          "batch_size": 16, "lr": 3e-5},
    "klue_roberta_large": {"hf_id": "klue/roberta-large",                         "batch_size": 8,  "lr": 1.5e-5},
    "koelectra":          {"hf_id": "monologg/koelectra-base-v3-discriminator",   "batch_size": 16, "lr": 3e-5},
}

CONFIG = {
    # 미션3은 텍스트만 쓰므로 라벨링(JSON)만 사용. val_audio_root는 inference.py CLI 호환용으로만 넘김
    "train_label_root": os.path.join(DRIVE_ROOT, "Training", "2.라벨링데이터", "TL_서울_구급"),
    "val_label_root":   os.path.join(DRIVE_ROOT, "Validation", "2.라벨링데이터", "VL_서울_구급"),
    "val_audio_root":   os.path.join(DRIVE_ROOT, "Validation", "1.원천데이터", "VS_서울_구급"),
    "local_json_dir": "/content/m3_local_json",

    # 팀 공유 Drive 폴더 (모델 폴더 / 확률 CSV / history 저장 위치)
    "drive_share_root": "/content/drive/MyDrive/DCC/mission3_v3",  # v0.2 결과와 분리. TODO: 팀 공유 폴더 경로 확인

    # 공식 9개 증상 목록을 받으면 여기에 리스트로 넣기. None이면 학습 데이터 빈도 상위 9개를 임시 사용(4번 셀 경고 참고)
    "symptom_labels": ["복통", "어지러움", "전신쇠약", "고열", "구토", "호흡곤란", "열상", "오심", "두통"],  # 팀 합의(기타 대신 두통), 공식 목록 여부는 미확인

    "use_speaker_tag": False,  # 팀 통일: 화자 태그 없이 발화 텍스트만 사용 (True로 바꾸면 [신고자]/[대원] 태그 추가, 효과 미검증)
    "max_length": 512,         # 초과분은 뒤쪽이 잘림. 4번 셀에서 잘림 비율 확인
    "train_frac": 1.0,         # 1.0 미만이면 학습 데이터 일부만 사용(백본 빠른 비교용). val은 항상 전체 사용
    "epochs": 5,
    "weight_decay": 0.01,
    "warmup_ratio": 0.06,
    "early_stop_patience": 2,
}

CONFIG["ckpt_dir"] = os.path.join(CONFIG["drive_share_root"], "ckpt")        # 모델 폴더(가중치+tokenizer+config)
CONFIG["preds_dir"] = os.path.join(CONFIG["drive_share_root"], "preds")
CONFIG["history_dir"] = os.path.join(CONFIG["drive_share_root"], "history")
for d in [CONFIG["ckpt_dir"], CONFIG["preds_dir"], CONFIG["history_dir"]]:
    os.makedirs(d, exist_ok=True)


Mounted at /content/drive
device=cuda, GPU=Tesla T4, torch=2.11.0+cu130, transformers=5.18.0


### 2. 라벨링(JSON) 데이터 로컬 복사

In [ ]:
# ===== JSON 전체를 로컬 디스크로 병렬 복사 (Drive 직접 읽기보다 훨씬 빠르고 안정적) =====
from concurrent.futures import ThreadPoolExecutor, as_completed

def copy_one_file(src_path, dst_path):
    if os.path.exists(dst_path) and os.path.getsize(dst_path) == os.path.getsize(src_path):
        return True, None
    tmp_path = f"{dst_path}.{uuid.uuid4().hex}.part"  # 스레드별 고유 임시 파일명 -> 이름 충돌 방지
    try:
        shutil.copy(src_path, tmp_path)
        os.rename(tmp_path, dst_path)
        return True, None
    except OSError as e:
        return False, str(e)

def copy_json_tree(src_root, dst_root, max_workers=16):
    os.makedirs(dst_root, exist_ok=True)
    tasks = []
    for dirpath, _, filenames in os.walk(src_root):
        rel_dir = os.path.relpath(dirpath, src_root)
        dst_dir = os.path.join(dst_root, rel_dir)
        os.makedirs(dst_dir, exist_ok=True)
        for fname in filenames:
            tasks.append((os.path.join(dirpath, fname), os.path.join(dst_dir, fname)))

    if len(tasks) == 0:
        raise RuntimeError(f"{src_root} 에서 파일을 하나도 찾지 못했습니다. 이 셀을 다시 실행해보세요.")

    print(f"{src_root}: {len(tasks)}개 파일 복사 시작 (병렬 {max_workers}개)")
    failed = []
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = {executor.submit(copy_one_file, s, d): s for s, d in tasks}
        for i, future in enumerate(as_completed(futures)):
            ok, err = future.result()
            if not ok:
                failed.append((futures[future], err))
            if (i + 1) % 5000 == 0:
                print(f"  {i+1}/{len(tasks)} 완료")

    if failed:
        print(f"복사 실패 {len(failed)}개 - 이 셀을 다시 실행하면 나머지만 이어서 복사합니다.")
    else:
        print(f"복사 완료: {len(tasks)}개 파일")

local_train_label = os.path.join(CONFIG["local_json_dir"], "train")
local_val_label = os.path.join(CONFIG["local_json_dir"], "val")
copy_json_tree(CONFIG["train_label_root"], local_train_label)
copy_json_tree(CONFIG["val_label_root"], local_val_label)


### 3. Manifest 생성 (대화 텍스트 + 증상 목록)

In [ ]:
# ===== 콜 하나 = JSON 하나 = 학습 샘플 하나 (대화 전체 텍스트 -> 증상 멀티라벨) =====
# symptom 필드는 '정답'이라 입력에 절대 넣지 않음. 입력은 발화 텍스트(+선택적으로 화자 태그)뿐.
SPEAKER_TAG = {1: "[신고자]", 0: "[대원]"}

def build_text(utterances, use_speaker_tag):
    utts = sorted(utterances, key=lambda u: u.get("startAt", 0))  # 시간순 정렬
    parts = []
    for u in utts:
        t = (u.get("text") or "").strip()
        if not t:
            continue
        if use_speaker_tag and u.get("speaker") in SPEAKER_TAG:
            t = f"{SPEAKER_TAG[u['speaker']]} {t}"
        parts.append(t)
    return " ".join(parts)

def parse_json(path, use_speaker_tag):
    with open(path, "r", encoding="utf-8") as f:
        meta = json.load(f)
    symptoms = meta.get("symptom") or []
    if isinstance(symptoms, str):
        symptoms = [symptoms]
    utterances = meta.get("utterances", meta.get("utterences", []))
    return {
        "file_name": os.path.basename(path),  # 제출 CSV의 'label file name'
        "text": build_text(utterances, use_speaker_tag),
        "symptoms": symptoms,
    }

def build_df(local_label_root, use_speaker_tag):
    rows = []
    for dirpath, _, filenames in os.walk(local_label_root):
        for fname in sorted(filenames):
            if fname.endswith(".json"):
                rows.append(parse_json(os.path.join(dirpath, fname), use_speaker_tag))
    return pd.DataFrame(rows)

train_df = build_df(local_train_label, CONFIG["use_speaker_tag"])
val_df = build_df(local_val_label, CONFIG["use_speaker_tag"])
print(f"train: {len(train_df)}건 (기대값 29,200), val: {len(val_df)}건 (기대값 3,640)")
print(f"텍스트가 빈 콜: train {(train_df['text'] == '').sum()}건, val {(val_df['text'] == '').sum()}건")
print("\n[샘플 텍스트]")
print(train_df.iloc[0]["text"][:300])
print("[샘플 증상]", train_df.iloc[0]["symptoms"])


### 4. 증상 카테고리 확정 + 분포/텍스트 길이 확인

In [ ]:
# ===== 증상 값 분포 확인 후 9개 카테고리 확정 =====
from collections import Counter

train_counter = Counter(s for lst in train_df["symptoms"] for s in lst)
val_counter = Counter(s for lst in val_df["symptoms"] for s in lst)
print(f"train 증상 값 종류: {len(train_counter)}개 / val 증상 값 종류: {len(val_counter)}개")
print("[train 상위 30]")
for name, n in train_counter.most_common(30):
    print(f"  {name}: {n} ({n / len(train_df):.1%})")

if CONFIG["symptom_labels"] is not None:
    SYMPTOM_LABELS = list(CONFIG["symptom_labels"])
elif len(train_counter) <= 9:
    SYMPTOM_LABELS = [n for n, _ in train_counter.most_common()]
else:
    SYMPTOM_LABELS = [n for n, _ in train_counter.most_common(9)]
    print("\n[경고] 증상 값이 9개를 넘어 학습 데이터 빈도 상위 9개를 임시로 사용합니다.")
    print("       공식 9개 목록/매핑을 확인하면 CONFIG['symptom_labels']에 넣고 1번 셀부터 다시 실행하세요.")
if len(SYMPTOM_LABELS) != 9:
    print(f"[경고] 증상 카테고리가 {len(SYMPTOM_LABELS)}개입니다 (문제 조건은 9개).")
print("\n사용할 증상 라벨:", SYMPTOM_LABELS)

missing = [l for l in SYMPTOM_LABELS if l not in train_counter]
if missing:
    print(f"[경고] 학습 데이터에 없는 라벨: {missing} (철자 확인)")
label_to_idx = {l: i for i, l in enumerate(SYMPTOM_LABELS)}

def to_vector(symptoms):
    v = np.zeros(len(SYMPTOM_LABELS), dtype=np.float32)
    for s in symptoms:
        if s in label_to_idx:
            v[label_to_idx[s]] = 1.0
    return v

train_df["y"] = train_df["symptoms"].apply(to_vector)
val_df["y"] = val_df["symptoms"].apply(to_vector)

Y_train = np.stack(train_df["y"].values)
print(f"\n콜당 평균 증상 수: {Y_train.sum(1).mean():.2f}, 사용 라벨이 하나도 없는 콜: {(Y_train.sum(1) == 0).mean():.1%}")
print("라벨별 양성 비율(train):", {l: round(float(Y_train[:, i].mean()), 3) for i, l in enumerate(SYMPTOM_LABELS)})

# ---- 텍스트 길이 확인 (max_length 초과 = 뒤쪽이 잘림) ----
default_tok = AutoTokenizer.from_pretrained(BACKBONES["kcelectra"]["hf_id"])
sample = train_df.sample(min(2000, len(train_df)), random_state=0)
lens = pd.Series([len(x) for x in default_tok(sample["text"].tolist())["input_ids"]])
print("\n[토큰 길이 분포]")
print(lens.describe(percentiles=[0.5, 0.9, 0.95]))
print(f"max_length={CONFIG['max_length']} 초과 비율: {(lens > CONFIG['max_length']).mean():.1%}")


### 5. Dataset / DataLoader

In [ ]:
# ===== 텍스트를 배치 단위로 토크나이즈(동적 패딩). 순서/화자 외 별도 특징은 쓰지 않음 =====
class CallTextDataset(Dataset):
    def __init__(self, df):
        self.texts = df["text"].tolist()
        self.y = np.stack(df["y"].values).astype(np.float32)
        self.files = df["file_name"].tolist()

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, i):
        return self.texts[i], self.y[i], self.files[i]


def make_collate(tokenizer, max_length):
    def collate(batch):
        texts, ys, files = zip(*batch)
        enc = tokenizer(list(texts), truncation=True, max_length=max_length, padding=True,
                        return_tensors="pt", return_token_type_ids=False)
        return dict(enc), torch.tensor(np.stack(ys)), list(files)
    return collate


### 6. 평가 · 학습 함수

In [ ]:
# ===== 공통 유틸 =====
THRESHOLD = 0.5            # 대회 규정: 모든 class에 0.5 고정 (튜닝 금지)
SELECT_METRIC = "f1_macro" # best epoch 선택 기준 (팀 통일: macro-F1, threshold와 무관)

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def metrics_from_pred(pred, y):
    """멀티라벨 지표. 'Accuracy'의 정의가 미확정이라 subset/hamming 둘 다 기록"""
    pred, y = pred.astype(int), y.astype(int)
    return {
        "f1_micro": float(f1_score(y, pred, average="micro", zero_division=0)),
        "f1_macro": float(f1_score(y, pred, average="macro", zero_division=0)),
        "subset_acc": float(accuracy_score(y, pred)),   # 9개 라벨이 전부 일치해야 정답
        "hamming_acc": float((pred == y).mean()),        # 라벨 하나하나의 평균 정확도
    }


def compute_metrics(probs, y):
    return metrics_from_pred((probs >= THRESHOLD).astype(int), y)  # threshold 0.5 고정


@torch.no_grad()
def predict(model, loader):
    model.eval()
    use_amp = device.type == "cuda"
    probs, ys, files, total_loss = [], [], [], 0.0
    for enc, y, f in loader:
        enc = {k: v.to(device) for k, v in enc.items()}
        with torch.amp.autocast("cuda", dtype=torch.float16, enabled=use_amp):
            logits = model(**enc).logits
        logits = logits.float().cpu()
        total_loss += F.binary_cross_entropy_with_logits(logits, y, reduction="sum").item() / y.shape[1]
        probs.append(torch.sigmoid(logits).numpy())
        ys.append(y.numpy())
        files.extend(f)
    n = sum(len(p) for p in probs)
    return np.concatenate(probs), np.concatenate(ys), files, total_loss / n


def save_atomic(obj, path):
    """임시 파일에 먼저 쓰고 rename -> 쓰는 도중 세션이 끊겨도 기존 체크포인트가 깨지지 않음"""
    tmp = path + ".tmp"
    torch.save(obj, tmp)
    os.replace(tmp, path)


def train_one_run(backbone_key, config_name, run_config, seed, config, train_df, val_df, labels):
    """
    백본 하나 x 설정 하나 x seed 하나를 학습하고 모델 폴더/history/확률 CSV/로그를 저장
    run_config: {"pos_weight": bool, "lr_scheduler": bool}
    이름 규칙: m3_{backbone}_{config_name}_s{seed}
    이어하기: 매 epoch 끝에 {이름}_last.pt 저장 -> 다시 실행하면 이어서 학습, 끝난 실행은 건너뜀
    """
    bb = BACKBONES[backbone_key]
    frac = config["train_frac"]
    tag = config_name + (f"_f{int(frac * 100)}" if frac < 1 else "")  # 일부 데이터 실험은 이름으로 구분
    full_name = f"m3_{backbone_key}_{tag}_s{seed}"
    model_dir = os.path.join(config["ckpt_dir"], f"{full_name}_best")
    last_path = os.path.join(config["ckpt_dir"], f"{full_name}_last.pt")
    history_path = os.path.join(config["history_dir"], f"{full_name}_history.csv")
    probs_path = os.path.join(config["preds_dir"], f"{full_name}_probs.csv")
    done_path = os.path.join(config["history_dir"], f"{full_name}_done.json")

    # ---- 이미 끝난 실행은 건너뜀 ----
    if os.path.exists(done_path):
        with open(done_path, "r", encoding="utf-8") as f:
            row = json.load(f)
        print(f"[{full_name}] 이미 완료된 실행이라 건너뜁니다: {row}")
        return row

    set_seed(seed)
    tokenizer = AutoTokenizer.from_pretrained(bb["hf_id"])
    model = AutoModelForSequenceClassification.from_pretrained(
        bb["hf_id"], num_labels=len(labels), problem_type="multi_label_classification"
    ).to(device)

    tr_df = train_df.sample(frac=frac, random_state=0) if frac < 1 else train_df
    bs, eval_bs = bb["batch_size"], bb["batch_size"] * 2
    collate = make_collate(tokenizer, config["max_length"])
    g = torch.Generator()  # epoch마다 시드를 다시 지정 -> 이어하기를 해도 데이터 순서가 동일
    train_loader = DataLoader(CallTextDataset(tr_df), batch_size=bs, shuffle=True, num_workers=2,
                              generator=g, collate_fn=collate)
    val_loader = DataLoader(CallTextDataset(val_df), batch_size=eval_bs, shuffle=False, num_workers=2,
                            collate_fn=collate)

    pos_weight = None
    if run_config["pos_weight"]:  # 희귀 증상 보완용 loss 가중치 (학습 설계일 뿐 threshold는 0.5 그대로)
        Y = np.stack(tr_df["y"].values)
        pos = Y.sum(0)
        pos_weight = torch.tensor(np.clip((len(Y) - pos) / np.maximum(pos, 1), 1.0, 10.0),
                                  dtype=torch.float32).to(device)
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

    optimizer = torch.optim.AdamW(model.parameters(), lr=bb["lr"], weight_decay=config["weight_decay"])
    total_steps = len(train_loader) * config["epochs"]
    scheduler = (get_linear_schedule_with_warmup(optimizer, int(total_steps * config["warmup_ratio"]), total_steps)
                 if run_config["lr_scheduler"] else None)
    use_amp = device.type == "cuda"
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)

    best_score, best_metrics, patience_counter = -1.0, None, 0
    history, start_epoch, prior_sec = [], 0, 0.0

    # ---- 이어하기: 마지막 epoch 상태가 있으면 복원 ----
    state = None
    if os.path.exists(last_path):
        try:
            state = torch.load(last_path, map_location=device, weights_only=False)
        except Exception as e:  # 저장 중 끊겨 파일이 깨진 경우 등
            print(f"[{full_name}] 이어하기 파일을 읽지 못해 처음부터 시작합니다: {e}")
    if state is not None:
        model.load_state_dict(state["model"])
        optimizer.load_state_dict(state["optimizer"])
        if scheduler is not None and state["scheduler"] is not None:
            scheduler.load_state_dict(state["scheduler"])
        scaler.load_state_dict(state["scaler"])
        history, best_score, best_metrics = state["history"], state["best_score"], state["best_metrics"]
        patience_counter, start_epoch, prior_sec = state["patience_counter"], state["epoch"], state["elapsed_sec"]
        print(f"[{full_name}] 이어서 시작: epoch {start_epoch + 1}부터 (이전 학습 {prior_sec / 60:.1f}분, best {SELECT_METRIC}={best_score:.4f})")
        del state

    t_start = time.time() - prior_sec  # 이전 학습 시간까지 누적해서 기록

    for epoch in range(start_epoch, config["epochs"]):
        if patience_counter >= config["early_stop_patience"]:
            print(f"[{full_name}] early stopping 조건이 이미 충족된 상태라 학습을 마칩니다")
            break
        set_seed(seed * 1000 + epoch)   # epoch별 고정 시드 (이어하기 해도 셔플/드롭아웃 재현)
        g.manual_seed(seed * 1000 + epoch)

        model.train()
        total_loss, t_ep = 0.0, time.time()
        for i, (enc, y, _) in enumerate(train_loader):
            enc = {k: v.to(device) for k, v in enc.items()}
            y = y.to(device)
            with torch.amp.autocast("cuda", dtype=torch.float16, enabled=use_amp):
                logits = model(**enc).logits
            loss = criterion(logits.float(), y)
            optimizer.zero_grad()
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            if scheduler is not None:
                scheduler.step()
            total_loss += loss.item() * y.size(0)
            if (i + 1) % 200 == 0:
                speed = (i + 1) / (time.time() - t_ep)
                print(f"    batch {i+1}/{len(train_loader)} | 이 에폭 남은 예상 {(len(train_loader) - i - 1) / speed / 60:.1f}분")
        train_loss = total_loss / len(tr_df)

        probs, y_true, files, val_loss = predict(model, val_loader)
        m = compute_metrics(probs, y_true)
        history.append({"epoch": epoch + 1, "train_loss": train_loss, "val_loss": val_loss, **m,
                        "lr": optimizer.param_groups[0]["lr"]})
        pd.DataFrame(history).to_csv(history_path, index=False)  # epoch마다 즉시 저장

        print(f"[{full_name}] epoch {epoch+1} train_loss={train_loss:.4f} val_loss={val_loss:.4f} "
              f"f1_micro={m['f1_micro']:.4f} f1_macro={m['f1_macro']:.4f} "
              f"subset_acc={m['subset_acc']:.4f} hamming_acc={m['hamming_acc']:.4f}")

        if m[SELECT_METRIC] > best_score:
            best_score, best_metrics, patience_counter = m[SELECT_METRIC], m, 0
            model.save_pretrained(model_dir)      # 가중치 + config
            tokenizer.save_pretrained(model_dir)  # tokenizer (오프라인 채점 대비, 폴더 통째로 제출)
            with open(os.path.join(model_dir, "meta.json"), "w", encoding="utf-8") as f:
                json.dump({"labels": labels, "max_length": config["max_length"],
                           "use_speaker_tag": config["use_speaker_tag"], "backbone": bb["hf_id"],
                           "threshold": THRESHOLD}, f, ensure_ascii=False)
            prob_df = pd.DataFrame(probs, columns=[f"prob_{l}" for l in labels])
            prob_df.insert(0, "call_id", [os.path.splitext(f)[0] for f in files])  # 확장자 뺀 파일명 (팀 약속)
            for j, l in enumerate(labels):
                prob_df[f"true_{l}"] = y_true[:, j].astype(int)
            prob_df.to_csv(probs_path, index=False)
        else:
            patience_counter += 1

        # ---- 매 epoch 끝: 이어하기 체크포인트 저장 (best 갱신/early stopping 카운트 반영 후) ----
        save_atomic({
            "model": model.state_dict(), "optimizer": optimizer.state_dict(),
            "scheduler": scheduler.state_dict() if scheduler is not None else None,
            "scaler": scaler.state_dict(), "history": history, "best_score": best_score,
            "best_metrics": best_metrics, "patience_counter": patience_counter,
            "epoch": epoch + 1, "elapsed_sec": time.time() - t_start,
        }, last_path)

        if patience_counter >= config["early_stop_patience"]:
            print(f"[{full_name}] early stopping (patience={config['early_stop_patience']})")
            break

    train_min = (time.time() - t_start) / 60

    # ---- best 모델을 다시 불러 Validation 전체 추론 시간/파라미터 수 측정 (제출 시 명시 권장 항목) ----
    del model, optimizer, scheduler
    torch.cuda.empty_cache()
    best_model = AutoModelForSequenceClassification.from_pretrained(model_dir).to(device)
    n_params = sum(p.numel() for p in best_model.parameters())
    if device.type == "cuda":
        torch.cuda.synchronize()
    t0 = time.time()
    predict(best_model, val_loader)
    if device.type == "cuda":
        torch.cuda.synchronize()
    infer_sec = time.time() - t0
    del best_model
    torch.cuda.empty_cache()

    row = {"backbone": backbone_key, "config": tag, "seed": seed,
           **{k: round(v, 4) for k, v in best_metrics.items()},
           "train_min": round(train_min, 1),
           "params_total_M": round(n_params / 1e6, 1),  # dense 모델이라 active = total
           "val_batch_size": eval_bs, "val_infer_sec": round(infer_sec, 1),
           "val_ms_per_sample": round(infer_sec / len(val_df) * 1000, 2), "gpu": GPU_NAME}
    print(f"[{full_name}] 완료 - {row}")

    log_path = os.path.join(config["drive_share_root"], "runs_log.csv")  # 팀 공통 로그에 한 줄 추가
    pd.DataFrame([row]).to_csv(log_path, mode="a", header=not os.path.exists(log_path), index=False)
    with open(done_path, "w", encoding="utf-8") as f:  # 완료 표시 (다시 실행하면 건너뜀)
        json.dump(row, f, ensure_ascii=False)
    if os.path.exists(last_path):
        os.remove(last_path)  # 이어하기 파일은 용량이 커서(모델+옵티마이저) 완료 후 삭제
    return row


### 7. 학습 실행

- **이어하기**: 같은 (백본, 설정, seed)를 다시 실행하면 `_last.pt`(마지막 epoch 상태)가 있을 때 거기서부터 이어서 학습합니다. 끊겨도 최대 1 epoch 분량만 잃습니다.
- **건너뛰기**: 끝난 실행은 `_done.json`이 남아 있어 다시 실행해도 학습하지 않고 결과만 반환합니다. 그래서 끊긴 뒤에는 이 셀을 그냥 다시 실행하면 됩니다.
- 처음부터 다시 하고 싶으면 Drive `ckpt/`의 `{이름}_last.pt`와 `history/`의 `{이름}_done.json`을 지우세요.


In [ ]:
# ===== 실행할 백본/설정 선택 =====
# 최종 제출 모델: KcELECTRA + "base"(pos_weight 없음), Training 전체(train_frac=1.0), seed 42/123/2026
EXPERIMENT_CONFIGS = {
    "base":      {"pos_weight": False, "lr_scheduler": True},
    "posweight": {"pos_weight": True,  "lr_scheduler": True},
}

BACKBONE_NAME = "kcelectra"   # 최종: kcelectra
CONFIG_NAME = "base"          # 최종: base (= pos_weight 없음)
RUN_SEEDS = SEEDS             # [42, 123, 2026]

if "all_results" not in globals():
    all_results = []  # 이번 세션 결과 누적 (세션이 끊겨도 Drive의 runs_log.csv에는 남음)

for seed in RUN_SEEDS:
    all_results.append(train_one_run(BACKBONE_NAME, CONFIG_NAME, EXPERIMENT_CONFIGS[CONFIG_NAME], seed,
                                     CONFIG, train_df, val_df, SYMPTOM_LABELS))

pd.DataFrame(all_results)

### 7b. 결과 기록 표 (팀 공통 로그)

In [3]:
# ===== 백본 / 설정 / seed별 결과 표 (Drive의 runs_log.csv 기준, 세션이 끊겨도 유지) =====
log_path = os.path.join(CONFIG["drive_share_root"], "runs_log.csv")
if os.path.exists(log_path):
    # 완료 직전에 끊겨 같은 실행이 두 번 기록됐을 수 있어 마지막 기록만 남김
    runs_log = (pd.read_csv(log_path).drop_duplicates(["backbone", "config", "seed"], keep="last")
                .sort_values(["backbone", "config", "seed"]).reset_index(drop=True))
else:
    runs_log = pd.DataFrame()
runs_log


,backbone,config,seed,f1_micro,f1_macro,subset_acc,hamming_acc,train_min,params_total_M,val_batch_size,val_infer_sec,val_ms_per_sample,gpu
0,kcelectra,base,42,0.6464,0.6083,0.4659,0.8998,60.2,127.8,32,27.5,7.55,Tesla T4
1,kcelectra,base,123,0.6456,0.6082,0.4626,0.8989,58.0,127.8,32,24.3,6.66,Tesla T4
2,kcelectra,base,2026,0.6474,0.6056,0.4577,0.9012,55.6,127.8,32,24.3,6.68,Tesla T4


### 7c. 학습 로그 (최종 3 seed)

학습 셀이 epoch마다 저장한 history CSV를 seed별로 출력합니다.

In [4]:
# 최종 모델 3개의 epoch별 학습 로그 (threshold 0.5 기준 Validation 지표)
for seed in SEEDS:
    name = f"m3_kcelectra_base_s{seed}"
    h = pd.read_csv(os.path.join(CONFIG["history_dir"], f"{name}_history.csv"))
    best = h.loc[h["f1_macro"].idxmax()]
    print(f"===== {name} | best macro-F1 {best['f1_macro']:.4f} (epoch {int(best['epoch'])}) =====")
    print(h[["epoch", "train_loss", "val_loss", "f1_macro", "f1_micro", "lr"]].round(4).to_string(index=False), "\n")

===== m3_kcelectra_base_s42 | best macro-F1 0.6083 (epoch 5) =====
 epoch  train_loss  val_loss  f1_macro  f1_micro  lr
     1      0.3367    0.2639    0.5808    0.6342 0.0
     2      0.2547    0.2569    0.5930    0.6425 0.0
     3      0.2345    0.2588    0.5922    0.6461 0.0
     4      0.2120    0.2700    0.6071    0.6485 0.0
     5      0.1919    0.2786    0.6083    0.6464 0.0 

===== m3_kcelectra_base_s123 | best macro-F1 0.6082 (epoch 5) =====
 epoch  train_loss  val_loss  f1_macro  f1_micro  lr
     1      0.3346    0.2684    0.5638    0.6235 0.0
     2      0.2552    0.2569    0.5849    0.6383 0.0
     3      0.2349    0.2601    0.5947    0.6472 0.0
     4      0.2129    0.2665    0.6040    0.6476 0.0
     5      0.1936    0.2755    0.6082    0.6456 0.0 

===== m3_kcelectra_base_s2026 | best macro-F1 0.6056 (epoch 5) =====
 epoch  train_loss  val_loss  f1_macro  f1_micro  lr
     1      0.3375    0.2693    0.5551    0.6145 0.0
     2      0.2591    0.2619    0.5770    0.6276 0

### 8. 성능 그래프

In [ ]:
# ===== Epoch별 Validation 지표 그래프 (저장된 history CSV 기준) =====
import matplotlib.pyplot as plt

PLOT_RUN_NAME = "m3_kcelectra_base_s42"  # 최종 모델 seed 42
h = pd.read_csv(os.path.join(CONFIG["history_dir"], f"{PLOT_RUN_NAME}_history.csv"))

plt.figure(figsize=(8, 4.5))
for col, marker in [("f1_micro", "o"), ("f1_macro", "s"), ("subset_acc", "^"), ("hamming_acc", "v")]:
    plt.plot(h["epoch"], h[col], marker=marker, label=col)
plt.xlabel("Epoch")
plt.ylabel("Score (threshold = 0.5)")
plt.title(f"Mission 3 - {PLOT_RUN_NAME} Validation metrics by Epoch")
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()
plt.savefig(f"/content/{PLOT_RUN_NAME}_metrics.png", dpi=150)
plt.show()